In [18]:
import pandas as pd
import pickle
import numpy as np 
import random
from tqdm import tqdm
from pathlib import Path
import re
from IPython.display import Audio

In [2]:
np.random.seed(0)


In [3]:
# file_out_dir = list(Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes/').glob('*dataset*.pdpkl'))
# Import timit data - this df contains all the relevant signals, labels and data 
path = '/om2/user/imgriff/datasets/timit/all_timit_wsn_compatible.pdpkl'
timit_excerpts = pd.read_pickle(path)


In [4]:
timit_excerpts = timit_excerpts.rename(columns={"_full_dataset_index":"_original_timit_index"})

In [5]:
# parse target & rest 

target_timit = timit_excerpts[timit_excerpts.word_int != 0]
# target_sentences = target_timit.sentence_id.unique()

# filter cues 
cue_timit = timit_excerpts[timit_excerpts.word_int == 0]
# cue_timit = timit_excerpts[(timit_excerpts.word_int == 0) & (~timit_excerpts.sentence_id.str.contains('1|2'))]


In [6]:
target_timit.reset_index(inplace=True, drop=True)

In [7]:
target_timit.shape

(902, 13)

## Define functions to be used with STRAIGHT

In [8]:
import scipy.signal
import scipy.interpolate
import scipy.io.wavfile
import librosa
import librosa.display
import matlab.engine
import sys 
sys.path.append('/om2/user/msaddler/python-packages/msutil')
import util_stimuli
import util_misc

def impose_inharmonic_jitter_pattern(y, sr, jitter, eng=None):
    x = matlab.double(y.reshape([-1, 1]).tolist())
    fs = matlab.double([sr])
    jitter_pattern = matlab.double(np.array(jitter).reshape([-1, 1]).tolist())
    
    y_resynth_matlab = eng.STRAIGHT_impose_inharmonic_jitter_pattern(
        x,
        fs,
        jitter_pattern)
    y_resynth = np.array(y_resynth_matlab).astype(np.float32).reshape([-1])
    NAN_IDX = np.isnan(y_resynth)
    if NAN_IDX.sum() > 0:
        print("Replacing {} NaN values (of {}) with zeros".format(NAN_IDX.sum(), NAN_IDX.shape[0]))
        y_resynth[NAN_IDX] = 0
    npad = int((y.shape[0] - y_resynth.shape[0]) / 2)
    y_resynth = np.pad(y_resynth, [npad, npad])
    return y_resynth



def make_whispered_speech(y, sr, fail_threshold=0.25, eng=None):
    y_resynth = eng.STRAIGHT_whispered_speech(
        matlab.double(y.reshape([-1, 1]).tolist()),
        matlab.double([sr]),
        nargout=1)
    y_resynth = np.array(y_resynth).astype(np.float32).reshape([-1])
    NAN_IDX = np.isnan(y_resynth)
    if NAN_IDX.sum() > 0:
        print("Replacing {} NaN values (of {}) with zeros".format(NAN_IDX.sum(), NAN_IDX.shape[0]))
        y_resynth[NAN_IDX] = 0
    y_resynth = util_stimuli.pad_or_trim_to_len(y_resynth, y.shape[0], mode='end')
    if (util_stimuli.rms(y_resynth) == 0) or (NAN_IDX.sum() / NAN_IDX.shape[0] > fail_threshold):
        print("STRAIGHT FAILURE --> returning input instead")
        y_resynth = y
    return y_resynth


def get_f0_contour(y, sr, eng=None):
    x = matlab.double(y.reshape([-1, 1]).tolist())
    fs = matlab.double([sr])
    f0_contour_matlab = eng.STRAIGHT_analysis_get_f0(x, fs)
    f0_contour = np.array(f0_contour_matlab).astype(np.float32).reshape([-1])
    NAN_IDX = np.isnan(f0_contour)
    if NAN_IDX.sum() > 0:
        print("Replacing {} NaN values (of {}) with zeros".format(NAN_IDX.sum(), NAN_IDX.shape[0]))
        f0_contour[NAN_IDX] = 0
    return f0_contour


In [9]:
eng = matlab.engine.start_matlab();
eng.addpath('/om2/user/imgriff/projects/msSTRAIGHT/Sinusoidal_Straight_Toolbox_v1.0/');
eng.addpath('/om2/user/imgriff/projects/msSTRAIGHT/');

In [10]:
y = target_timit['signal'].iloc[0]

In [11]:
f0s = get_f0_contour(y, 20_000, eng=eng)

Elapsed time is 0.851976 seconds.
Elapsed time is 0.141864 seconds.
Elapsed time is 0.378735 seconds.
Elapsed time is 0.884671 seconds.


## Making Inharmonic Speech Dataset 

Use only single-distractors.  We are trying to emulate the concurent sentences experiment (experiment 5) in the Popham et al. 2018 paper. The paper doesn't state the SNR used in this experiement, so we will start with setting target and distractor to the same level (0dB). 

Proceedure for generating each set of trial stimuli:
 * select cue, target, and distractor signals
 * get min f0 over all three signals
 * create jitter pattern
 * resynth all three signals 
 * mix target and distractor 
 * save all inharmonic signals (cue, target, distractor, and mixture)
 
 
#### Note:  could also include a few other levels for more data on performance (-3, +3 dB SNR) at a later data.


___ 
### Find bug that happens on certain trials 

Creating jitter pattern runs infinite loop - lets find out why 

In [46]:
eg_bad_ix = (7 * 50) + 5

print(eg_bad_ix)

355


### demo of single trial:

In [48]:
target = target_timit['signal'].iloc[eg_bad_ix]
speaker = target_timit['speaker'].iloc[eg_bad_ix]

cue = cue_timit[cue_timit.speaker == speaker].sample(1)['signal'].item()

distractor_data = target_timit[target_timit.speaker!=speaker].sample(1)
distractor = distractor_data['signal'].item()

min_f0s = []

# want min of means 
for source in [cue, target, distractor]:
    f0_contour = get_f0_contour(source, 20_000, eng=eng)
    min_contour_f0 = f0_contour[f0_contour > 0].min()
    min_f0s.append(min_contour_f0)

# tmp hack to avoid bad jitter: clip f0s below lowest point  
min_f0 = np.array(min_f0s).clip(60).min()
print(min_f0s, min_f0)
# use params from Popham et al. 2018
harm_nums = np.arange(1, 31)
jitter_amount = 0.3
min_diff = 30.0

jitter_pattern = eng.make_jittered_speech_harmonics(
    matlab.double([min_f0]),
    matlab.double(harm_nums.tolist()),
    matlab.double([jitter_amount]),
    matlab.double([min_diff]))

# jitter_pattern = np.array(jitter_pattern).reshape([-1]).astype(float)

# cue_inharm = impose_inharmonic_jitter_pattern(cue, 20_000, jitter_pattern, eng=eng)
# target_inharm = impose_inharmonic_jitter_pattern(target, 20_000, jitter_pattern, eng=eng)
# distractor_inharm = impose_inharmonic_jitter_pattern(distractor, 20_000, jitter_pattern, eng=eng)

Elapsed time is 0.341004 seconds.
Elapsed time is 0.027170 seconds.
Elapsed time is 0.324849 seconds.
Elapsed time is 0.448008 seconds.
Elapsed time is 0.342154 seconds.
Elapsed time is 0.027063 seconds.
Elapsed time is 0.362507 seconds.
Elapsed time is 0.445916 seconds.
Elapsed time is 0.340601 seconds.
Elapsed time is 0.027281 seconds.
Elapsed time is 0.334926 seconds.
Elapsed time is 0.454866 seconds.
[73.40202, 64.57564, 76.10332] 64.57564


In [55]:


jitter_pattern = eng.make_jittered_speech_harmonics(
    matlab.double([0]),
    matlab.double(harm_nums.tolist()),
    matlab.double([0]),
    matlab.double([0]))
np.array(jitter_pattern).reshape([-1]).astype(float).shape

(30,)

In [57]:
np.zeros(30)

(30,)

In [43]:
jitter_pattern = eng.make_jittered_speech_harmonics(
    matlab.double([min_f0]),
    matlab.double(harm_nums.tolist()),
    matlab.double([jitter_amount]),
    matlab.double([min_diff]))

the MATLAB function has been cancelled


Operation terminated by user during make_jittered_speech_harmonics (line 23)



In [30]:
Audio(distractor, rate=20_000, normalize=False)

In [34]:
y_resynth = impose_inharmonic_jitter_pattern(y, 20_000, jitter_pattern, eng=eng)

Elapsed time is 0.403606 seconds.
Elapsed time is 0.030503 seconds.
Elapsed time is 0.163040 seconds.
Elapsed time is 0.825320 seconds.


In [41]:
util_stimuli.rms(y_resynth), util_stimuli.rms(y)

(0.14777178, 0.10000000000000002)

In [40]:
from IPython.display import Audio

Audio(y_resynth, rate=20000, normalize=True)

In [50]:
y_whispered = make_whispered_speech(y, 20_000, eng=eng)

Elapsed time is 0.339474 seconds.
Elapsed time is 0.025442 seconds.
Elapsed time is 0.166376 seconds.
Elapsed time is 0.453419 seconds.


In [66]:
next_pow_2 = lambda x: int(pow(2, np.ceil(np.log2(x))))

def combine_with_noise(clean, noise, snr):
    # get ratio in rms 
    rms_ratio = np.power(10.0, snr / 20.0)
    
    # remove DC of each signal
    clean = clean - clean.mean()
    noise = noise - noise.mean()
    # get rms of each signal
    clean_rms = np.sqrt(np.mean(np.power(clean, 2)))
    noise_rms = np.sqrt(np.mean(np.power(noise, 2)))
    # scale factor for setting noise to desired SNR 
    scale_factor = clean_rms / (noise_rms * rms_ratio)
    # Blend signals 
    noise = noise * scale_factor
    mixture = clean + noise[:len(clean)]
    return mixture, scale_factor

def rms_normalize(wav, new_rms=0.1, axis=0): 
    wav = wav - wav.mean(axis=axis)
    rms_wav = np.sqrt(np.mean(np.power(wav, 2), axis=axis))
    scale_factor = new_rms / rms_wav
    wav = wav * scale_factor
    return wav, scale_factor

## Define windowing function - will apply a cosine ramp to the start and end of a signal

In [67]:
def update_dict(dict_to_update, dict_with_vals):
    for key,value in dict_with_vals.items():
        if key not in dict_to_update:
            dict_to_update[key] = [value]
        else:
            dict_to_update[key].append(value)
            

In [80]:
row

word                                                                should
_original_timit_index                                                   28
source                                               train-dr1-fdml0-sx429
speaker                                                              fdml0
sr                                                                   20000
signal_length                                                        40000
speaker_sex                                                              f
sentence_type                                                           sx
sentence_id                                                            429
dialect_region                                                         dr1
data_split                                                           train
signal                   [0.0026449233, 0.009264113, 0.012351385, 0.005...
word_int                                                               646
Name: 38, dtype: object

## Below is example of loop used to create stimuli

This loop takes too long to run (~8 hours). To speed things up, we parallelized stimuli generation using `src/make_inharm_timit.py` and `submit_make_inharm_timit.sh` 

To get the final dataset, concatenate the stimuli into one df

In [133]:
# Generate trial by trial stim    
trial_data = {'signal':[],
              'target_signal':[],
              'speaker': [],
              'speaker_sex': [],
              'sentence_type': [],
              'word_int': [],
              'mixture_signal':[],
              'cue_signal': [],
              'cue_speaker': [],
              'cue_word': [],
              'distractor_signal':[],
              '_original_distractor_timit_indices':[],
              '_original_cue_timit_index':[],
              'distractor_words':[],
              'distractor_speakers':[],
              'distractor_conditions':[],
              'distractor_sex':[],
              'snrs':[]}


# use params from Popham et al. 2018
harm_nums = np.arange(1, 31)
jitter_amount = 0.3
min_diff = 30.0

snr = 0 # start with 0 dB 
for ix, row in tqdm(target_timit.iloc[50:52].iterrows()):
    # add wantned data to trial dict
    update_dict(trial_data, row)
    
    # get signals 
    target_signal = row.signal
    
    distractor_data = target_timit[target_timit.speaker!=row.speaker].sample(1)
    distractor_signal = distractor_data['signal'].item()
    
    cue_data = cue_timit[cue_timit.speaker == row.speaker].sample(1)
    cue_signal = cue_data['signal'].item()
    
    ## get min f0s 
    min_f0s = []
    for source in [cue, target, distractor]:
        f0_contour = get_f0_contour(source, 20_000, eng=eng)
        min_contour_f0 = f0_contour[f0_contour!=0].min()
        min_f0s.append(min_contour_f0)
    
    min_f0 = min(min_f0s)
    
    # Make harmonic jitter 
    jitter_pattern = eng.make_jittered_speech_harmonics(
        matlab.double([min_f0]),
        matlab.double(harm_nums.tolist()),
        matlab.double([jitter_amount]),
        matlab.double([min_diff]))

    jitter_pattern = np.array(jitter_pattern).reshape([-1]).astype(float)
    
    # get inharmonic signals 
    cue_inharm = impose_inharmonic_jitter_pattern(cue, 20_000, jitter_pattern, eng=eng)
    target_inharm = impose_inharmonic_jitter_pattern(target, 20_000, jitter_pattern, eng=eng)
    distractor_inharm = impose_inharmonic_jitter_pattern(distractor, 20_000, jitter_pattern, eng=eng) 
    
    # mix target & distractor
    mixture, _ = combine_with_noise(target_inharm, distractor_inharm, snr) # first_scale_factor unused
    mixture, mixture_scale_factor = rms_normalize(mixture)
    
    cue, _ = rms_normalize(cue_inharm)
    # rove cue
    cue = cue * mixture_scale_factor
    
    # save values for tiral 
    trial_data['signal'][ix%50] = target_inharm
    trial_data['mixture_signal'].append(mixture)
    trial_data['distractor_signal'].append(distractor_inharm)
    trial_data['cue_signal'].append(cue)
    trial_data['cue_word'].append(cue_data.word.item())
    trial_data['cue_speaker'].append(cue_data.speaker.item())
    trial_data['_original_cue_timit_index'].append(cue_data._original_timit_index.item())
    trial_data['_original_distractor_timit_indices'].append(distractor_data._original_timit_index.item())
    trial_data['distractor_words'].append(distractor_data.word.item())
    trial_data['distractor_speakers'].append(distractor_data.speaker.item())
    trial_data['distractor_conditions'].append('inharmonic')
    trial_data['distractor_sex'].append(distractor_data.speaker_sex.item())
    trial_data['snrs'].append(snr)
#     if ix == 1:
#         break


0it [00:00, ?it/s]

Elapsed time is 0.328082 seconds.
Elapsed time is 0.024272 seconds.
Elapsed time is 0.249840 seconds.
Elapsed time is 0.450548 seconds.
Elapsed time is 0.327998 seconds.
Elapsed time is 0.019380 seconds.
Elapsed time is 0.160952 seconds.
Elapsed time is 0.442677 seconds.
Elapsed time is 0.323601 seconds.
Elapsed time is 0.020203 seconds.
Elapsed time is 0.243218 seconds.
Elapsed time is 0.438343 seconds.
Elapsed time is 0.326673 seconds.
Elapsed time is 0.023489 seconds.
Elapsed time is 0.251060 seconds.
Elapsed time is 0.434243 seconds.
Elapsed time is 0.326550 seconds.
Elapsed time is 0.020164 seconds.
Elapsed time is 0.238172 seconds.
Elapsed time is 0.443056 seconds.
Elapsed time is 0.325675 seconds.
Elapsed time is 0.020718 seconds.
Elapsed time is 0.243002 seconds.
Elapsed time is 0.447724 seconds.


1it [00:37, 37.26s/it]

Elapsed time is 0.326134 seconds.
Elapsed time is 0.023680 seconds.
Elapsed time is 0.252655 seconds.
Elapsed time is 0.429874 seconds.
Elapsed time is 0.322451 seconds.
Elapsed time is 0.019904 seconds.
Elapsed time is 0.161196 seconds.
Elapsed time is 0.441062 seconds.
Elapsed time is 0.324062 seconds.
Elapsed time is 0.020174 seconds.
Elapsed time is 0.241998 seconds.
Elapsed time is 0.439775 seconds.
Elapsed time is 0.323969 seconds.
Elapsed time is 0.023124 seconds.
Elapsed time is 0.239518 seconds.
Elapsed time is 0.431318 seconds.
Elapsed time is 0.326341 seconds.
Elapsed time is 0.019858 seconds.
Elapsed time is 0.162744 seconds.
Elapsed time is 0.440338 seconds.
Elapsed time is 0.325043 seconds.
Elapsed time is 0.019981 seconds.
Elapsed time is 0.243443 seconds.
Elapsed time is 0.450309 seconds.


2it [01:13, 36.95s/it]


In [135]:
{key:len(vals) for key,vals in trial_data.items()}

{'signal': 2,
 'speaker': 2,
 'speaker_sex': 2,
 'sentence_type': 2,
 'word_int': 2,
 'mixture_signal': 2,
 'cue_signal': 2,
 'cue_speaker': 2,
 'cue_word': 2,
 'distractor_signal': 2,
 '_original_distractor_timit_indices': 2,
 '_original_cue_timit_index': 2,
 'distractor_words': 2,
 'distractor_speakers': 2,
 'distractor_conditions': 2,
 'distractor_sex': 2,
 'snrs': 2,
 'word': 2,
 '_original_timit_index': 2,
 'source': 2,
 'sr': 2,
 'signal_length': 2,
 'sentence_id': 2,
 'dialect_region': 2,
 'data_split': 2}

In [134]:
model_data = pd.DataFrame.from_dict(trial_data)

In [118]:
word_and_speaker_encodings = pickle.load( open( "/om2/user/imgriff/projects/End-to-end-ASR-Pytorch/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
class_map = word_and_speaker_encodings['word_idx_to_word']
word_to_ix_map = {val:key for key,val in class_map.items()}



In [56]:
def get_ix_from_words(words):
    if not isinstance(words, list):
        return words
    else:
        return [word_to_ix_map[word] for word in words]

model_timit_df['distractor_word_ints'] = model_timit_df['distractor_words'].apply(get_ix_from_words)


In [57]:
# model_timit_df.columns

Index(['signal', 'speaker', 'speaker_sex', 'sentence_type', 'word_int',
       'mixture_signal', 'cue_signal', 'cue_speaker', 'cue_word',
       '_original_distractor_timit_indices', '_original_cue_timit_index',
       'distractor_words', 'distractor_speakers', 'distractor_conditions',
       'distractor_sex', 'snrs', 'word', '_original_timit_index', 'source',
       'sr', 'signal_length', 'sentence_id', 'dialect_region', 'data_split',
       'distractor_word_ints'],
      dtype='object')

In [58]:
# out_path = Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes/')

# model_timit_df.to_pickle(out_path / 'timit_attn_stim_for_model_all_targets.pdpkl')

In [3]:
# f_path = '/om2/user/imgriff/datasets/timit/attn_task_dataframes/timit_attn_stim_for_model_all_targets.pdpkl'
# timit_df = pd.read_pickle(f_path)


In [4]:
timit_df.columns

Index(['signal', 'speaker', 'speaker_sex', 'sentence_type', 'word_int',
       'mixture_signal', 'cue_signal', 'cue_speaker', 'cue_word',
       '_original_distractor_timit_indices', '_original_cue_timit_index',
       'distractor_words', 'distractor_speakers', 'distractor_conditions',
       'distractor_sex', 'snrs', 'word', '_original_timit_index', 'source',
       'sr', 'signal_length', 'sentence_id', 'dialect_region', 'data_split',
       'distractor_word_ints'],
      dtype='object')

In [6]:
from IPython.display import Audio


In [116]:
Audio(cue_inharm, rate=20000)

## Making Whisper Speech Dataset 


Same idea as the inharmonic but now with whisper speech, which does not require the min f0 computation.  

Use only single-distractors.  We are trying to emulate the concurent sentences experiment (experiment 5) in the Popham et al. 2018 paper. The paper doesn't state the SNR used in this experiement, so we will start with setting target and distractor to the same level (0dB). 

Proceedure for generating each set of trial stimuli:
 * select cue, target, and distractor signals
 * resynth all three signals as whispers 
 * mix target and distractor 
 * save all whisper signals (cue, target, distractor, and mixture)
 
 
#### Note:  could also include a few other levels for more data on performance (-3, +3 dB SNR) at a later data.


### Demo of generation for single trial 

In [102]:
target = target_timit['signal'].iloc[0]
speaker = target_timit['speaker'].iloc[0]

cue = cue_timit[cue_timit.speaker == speaker].sample(1)['signal'].item()

distractor_data = target_timit[target_timit.speaker!=speaker].sample(1)
distractor = distractor_data['signal'].item()

sr = 20_000

cue_whisper = make_whispered_speech(cue, sr, eng=eng)
target_whisper = make_whispered_speech(target, sr, eng=eng)
distractor_whisper = make_whispered_speech(distractor, sr, eng=eng)

Elapsed time is 0.323144 seconds.
Elapsed time is 0.020084 seconds.
Elapsed time is 0.156435 seconds.
Elapsed time is 0.445194 seconds.
Elapsed time is 0.324262 seconds.
Elapsed time is 0.019516 seconds.
Elapsed time is 0.166315 seconds.
Elapsed time is 0.443577 seconds.
Elapsed time is 0.335011 seconds.
Elapsed time is 0.021168 seconds.
Elapsed time is 0.253425 seconds.
Elapsed time is 0.437730 seconds.


### signals generated for all stim in scripts

Like the inharmonnic stimuli generation, all stimuli were made using `src/make_whisper_timit.py`, submitted as job arrays with `submit_make_whisper_timit.sh` 

## Make Clean only stim for normal speech 

In [2]:
all_timit = pd.read_pickle('/om2/user/imgriff/datasets/timit/attn_task_dataframes/timit_attn_stim_for_model_all_targets.pdpkl')

In [3]:
all_timit.head()

,signal,speaker,speaker_sex,sentence_type,word_int,mixture_signal,cue_signal,cue_speaker,cue_word,_original_distractor_timit_indices,...,snrs,word,_original_timit_index,source,sr,signal_length,sentence_id,dialect_region,data_split,distractor_word_ints
0,"[0.00012005048285174084, 0.0010252486123257909...",fdaw0,f,sx,552,"[0.09473495656716215, 0.07726425141867858, 0.0...","[-6.021130907082137e-05, 0.0002179752211393519...",fdaw0,slowed,[5748],...,-6,programs,15,train-dr1-fdaw0-sx146,20000,40000,146,dr1,train,[working]
1,"[0.011782246730868976, 0.08233421110820222, 0....",fdaw0,f,sx,461,"[-0.020054463556352937, 0.010147070949337084, ...","[1.686975271292467e-05, -0.0001628872552172854...",fdaw0,answer,[6095],...,-6,novel,17,train-dr1-fdaw0-sx326,20000,40000,326,dr1,train,[medical]
2,"[0.0026449234716838257, 0.009264113281959791, ...",fdml0,f,sx,646,"[0.002029729771053116, 0.004756987291048679, 0...","[0.00028486117086427246, -0.000579473807245304...",fdml0,tube,[978],...,-6,should,28,train-dr1-fdml0-sx429,20000,40000,429,dr1,train,[larger]
3,"[-0.00037831780339846264, -0.00045162531152865...",fecd0,f,sx,659,"[-0.018556469254813943, -0.007759291164789502,...","[-0.012412901495380104, -0.015420376258757239,...",fecd0,wound,[1679],...,-6,social,35,train-dr1-fecd0-sx158,20000,40000,158,dr1,train,[caused]
4,"[0.029901661481428206, 0.01862221753076525, 0....",fecd0,f,sx,393,"[0.12764520140291172, 0.0784103245386344, 0.02...","[-0.025108234638327256, -0.027701923768734996,...",fecd0,to,[342],...,-6,light,36,train-dr1-fecd0-sx248,20000,40000,248,dr1,train,[young]


In [29]:
clean_timit = all_timit.loc[(all_timit['distractor_conditions'] == 'ssn')
                            & (all_timit['snrs'] == 0)
                            ].reset_index(drop=True)

array([-0.0723974 , -0.08332606, -0.07563086, ..., -0.00727153,
       -0.00723461, -0.00474659])

In [33]:
clean_timit.to_pickle('/om2/user/imgriff/datasets/timit/clean_timit_targets_attn_task_0.1rms.pdpkl')

## Eg on making safe & nomralized timit 

In [ ]:
# file_out_dir = list(Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes/').glob('*dataset*.pdpkl'))
# Import timit data - this df contains all the relevant signals, labels and data 
path = '/om2/user/imgriff/datasets/timit/all_timit_wsn_compatible.pdpkl'
timit_excerpts = pd.read_pickle(path)


In [43]:
## Made in timit_screen.ipynb on local desktop 

bad_sentences = ['sx74', 'sx144', 'sx376', 'si615', 'si670', 'si688', 'si693',
       'si917', 'si926', 'si927', 'si930', 'si931', 'si1078', 'si1102',
       'si1106', 'si1241', 'si1616', 'si1739', 'si1752', 'si1758',
       'si1767', 'si1933', 'si2020', 'si2040', 'si2064', 'si2112',
       'si2134', 'si2204', 'si2284']


timit_excerpts = timit_excerpts[~timit_excerpts.sentence_id.isin(bad_sentences)]

In [44]:
## Normalize all audio signals 
all_signals = np.stack(timit_excerpts.signal).astype('float32')

all_signals = all_signals - all_signals.mean(1).reshape(-1,1) # de-mean rows 


per_signal_rms = np.sqrt(np.mean(np.power(all_signals, 2), axis=1)).reshape(-1,1) # per signal rms 


all_signals = all_signals * 0.1 / per_signal_rms # normalize all signals to rms of 0.02 or ~60dB

timit_excerpts['signal'] = [signal for signal in all_signals] # update signal column

In [14]:
timit_excerpts.to_pickle('/om2/user/imgriff/datasets/timit/safe_sentences_timit_wsn_compatible_0.1rms.pdpkl')

NameError: name 'timit_excerpts' is not defined

In [88]:
timit_excerpts.columns

Index(['word', '_original_timit_index', 'source', 'speaker', 'sr',
       'signal_length', 'speaker_sex', 'sentence_type', 'sentence_id',
       'dialect_region', 'data_split', 'signal', 'word_int'],
      dtype='object')